In [5]:
"""
Heart Disease Prediction — Naive Bayes
=======================================
Network structure (from PROJECT_REPORT.docx):

  Layer 1 (Root):      age, sex
  Layer 2 (Risk):      chol, thalach, ca   ← depend on age, sex
  Layer 3 (Symptoms):  cp ← chol
                       exang ← thalach
                       oldpeak ← thalach
  Layer 4 (Target):    target ← cp, exang, oldpeak

Naive Bayes assumption used here:
  P(Y | X1..X8) ∝ P(Y) * ∏ P(Xi | Y)
  i.e. all 8 features are treated as conditionally independent given Y.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)
from sklearn.naive_bayes import GaussianNB
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0.  DATA LOADING
# ─────────────────────────────────────────────
# Using the UCI Heart Disease dataset (Cleveland).
# Download automatically via URL if not present.
import os, urllib.request

DATA_URL = ("https://raw.githubusercontent.com/dsrscientist/"
            "dataset1/master/heart.csv")
DATA_PATH = "heart_disease.csv"

if not os.path.exists(DATA_PATH):
    print("Downloading heart.csv …")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)

# ── Standardise column names (handles both naming conventions) ──
rename_map = {
    "HeartDisease": "target",
    "MaxHR": "thalach",
    "Cholesterol": "chol",
    "Age": "age",
    "Sex": "sex",
    "ChestPainType": "cp",
    "FastingBS": "fbs",
    "RestingECG": "restecg",
    "ExerciseAngina": "exang",
    "Oldpeak": "oldpeak",
    "ST_Slope": "slope",
}
df.rename(columns=rename_map, inplace=True)

# Encode string categoricals that appear in the Kaggle version
for col in ["sex", "cp", "exang"]:
    if df[col].dtype == object:
        df[col] = df[col].astype("category").cat.codes

# Keep only the 8 features from the report + target
FEATURES = ["age", "sex", "cp", "chol", "thalach", "exang", "oldpeak", "ca"]
missing = [c for c in FEATURES + ["target_binary"] if c not in df.columns]
if missing:
    raise ValueError(f"Columns not found in dataset: {missing}\n"
                     f"Available: {list(df.columns)}")

df = df[FEATURES + ["target_binary"]].dropna()
print(f"\nDataset shape: {df.shape}")
print(f"Class distribution:\n{df['target_binary'].value_counts().to_string()}\n")

X = df[FEATURES].values
y = df["target_binary"].values

# ── Label types: which features are categorical vs numerical ──
# age=num, sex=cat, cp=cat, chol=num, thalach=num, exang=cat, oldpeak=num, ca=cat
CATEGORICAL_IDX = [1, 2, 5, 7]   # sex, cp, exang, ca
NUMERICAL_IDX   = [0, 3, 4, 6]   # age, chol, thalach, oldpeak

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


Dataset shape: (1024, 9)
Class distribution:
target_binary
0    554
1    470



In [6]:
# ═══════════════════════════════════════════════════════════════
# PART 1 — FROM SCRATCH  (Mixed Naive Bayes)
# Gaussian likelihood for numerical features,
# Categorical (multinomial) likelihood for discrete features.
# ═══════════════════════════════════════════════════════════════
 
class MixedNaiveBayesScratch:
    """
    Naive Bayes that handles both Gaussian (numerical) and
    Categorical (discrete) features.
 
    P(Y=c | X) ∝ P(Y=c) * ∏_num  GaussianPDF(Xi | μ_c, σ_c)
                          * ∏_cat P(Xi=xi | Y=c)   [Laplace smoothed]
    """
 
    def __init__(self, cat_idx, num_idx, alpha=1.0):
        self.cat_idx = cat_idx        # column indices of categorical features
        self.num_idx = num_idx        # column indices of numerical features
        self.alpha   = alpha          # Laplace smoothing for categorical
 
    # ── helpers ──────────────────────────────────────────────
    @staticmethod
    def _gaussian_pdf(x, mean, var):
        """Gaussian probability density (with variance floor for stability)."""
        var = np.maximum(var, 1e-9)
        coeff = 1.0 / np.sqrt(2 * np.pi * var)
        exponent = np.exp(-0.5 * ((x - mean) ** 2) / var)
        return coeff * exponent
 
    # ── fit ──────────────────────────────────────────────────
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        n_samples = len(y)
 
        # ── prior probabilities ──
        self.log_prior_ = {}
        for c in self.classes_:
            self.log_prior_[c] = np.log(np.sum(y == c) / n_samples)
 
        # ── Gaussian params for numerical features ──
        self.gauss_params_ = {}           # {class: {feat_idx: (mean, var)}}
        for c in self.classes_:
            X_c = X[y == c]
            self.gauss_params_[c] = {}
            for idx in self.num_idx:
                mean = X_c[:, idx].mean()
                var  = X_c[:, idx].var()
                self.gauss_params_[c][idx] = (mean, var)
 
        # ── Categorical likelihoods with Laplace smoothing ──
        self.cat_probs_ = {}              # {class: {feat_idx: {value: log_p}}}
        for c in self.classes_:
            X_c = X[y == c]
            self.cat_probs_[c] = {}
            for idx in self.cat_idx:
                values, counts = np.unique(
                    X[:, idx].astype(int), return_counts=True)  # universe of values
                n_vals = len(values)
                n_c    = X_c.shape[0]
                self.cat_probs_[c][idx] = {}
                for v in values:
                    count_v = np.sum(X_c[:, idx].astype(int) == v)
                    # Laplace smoothed probability
                    p = (count_v + self.alpha) / (n_c + self.alpha * n_vals)
                    self.cat_probs_[c][idx][v] = np.log(p)
        return self
 
    # ── predict ──────────────────────────────────────────────
    def _log_likelihood(self, x, c):
        log_like = 0.0
 
        # Gaussian terms
        for idx in self.num_idx:
            mean, var = self.gauss_params_[c][idx]
            pdf = self._gaussian_pdf(x[idx], mean, var)
            log_like += np.log(pdf + 1e-300)   # clip to avoid log(0)
 
        # Categorical terms
        for idx in self.cat_idx:
            v = int(x[idx])
            log_p = self.cat_probs_[c][idx].get(
                v, np.log(self.alpha / (1 + self.alpha)))   # unseen value
            log_like += log_p
 
        return log_like
 
    def predict_proba(self, X):
        log_posts = np.array([
            [self.log_prior_[c] + self._log_likelihood(x, c)
             for c in self.classes_]
            for x in X
        ])
        # Normalise in log space → softmax
        log_posts -= log_posts.max(axis=1, keepdims=True)
        proba = np.exp(log_posts)
        proba /= proba.sum(axis=1, keepdims=True)
        return proba
 
    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]
 
 
# ── Train & evaluate from-scratch model ─────────────────────────────────────
print("=" * 60)
print("PART 1 — NAIVE BAYES FROM SCRATCH (Mixed: Gaussian + Categorical)")
print("=" * 60)
 
scratch_model = MixedNaiveBayesScratch(
    cat_idx=CATEGORICAL_IDX,
    num_idx=NUMERICAL_IDX,
    alpha=1.0
)
scratch_model.fit(X_train, y_train)
y_pred_scratch = scratch_model.predict(X_test)
 
print(f"\nAccuracy  : {accuracy_score(y_test, y_pred_scratch):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_scratch,
                            target_names=["No Disease", "Disease"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_scratch))
 
# ── Inspect learned parameters ──────────────────────────────────────────────
print("\n── Gaussian Parameters (mean | std) per class ──")
feat_names = ["age", "chol", "thalach", "oldpeak"]
for c in scratch_model.classes_:
    print(f"\n  Class {c}:")
    for idx, name in zip(NUMERICAL_IDX, feat_names):
        mean, var = scratch_model.gauss_params_[c][idx]
        print(f"    {name:10s}: mean={mean:.2f}, std={np.sqrt(var):.2f}")
 
print("\n── Categorical Likelihoods P(Xi | Y) ──")
cat_feat_names = {1: "sex", 2: "cp", 5: "exang", 7: "ca"}
for c in scratch_model.classes_:
    print(f"\n  Class {c}:")
    for idx, name in cat_feat_names.items():
        probs = {v: np.exp(lp)
                 for v, lp in scratch_model.cat_probs_[c][idx].items()}
        prob_str = "  ".join(f"P({name}={v})={p:.3f}"
                             for v, p in sorted(probs.items()))
        print(f"    {prob_str}")

PART 1 — NAIVE BAYES FROM SCRATCH (Mixed: Gaussian + Categorical)

Accuracy  : 0.8585

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.87      0.87      0.87       111
     Disease       0.85      0.84      0.84        94

    accuracy                           0.86       205
   macro avg       0.86      0.86      0.86       205
weighted avg       0.86      0.86      0.86       205

Confusion Matrix:
[[97 14]
 [15 79]]

── Gaussian Parameters (mean | std) per class ──

  Class 0:
    age       : mean=52.49, std=10.11
    chol      : mean=242.58, std=56.89
    thalach   : mean=158.09, std=19.30
    oldpeak   : mean=0.65, std=0.67

  Class 1:
    age       : mean=56.60, std=7.53
    chol      : mean=253.41, std=48.02
    thalach   : mean=138.97, std=22.31
    oldpeak   : mean=1.63, std=1.21

── Categorical Likelihoods P(Xi | Y) ──

  Class 0:
    P(sex=0)=0.436  P(sex=1)=0.564
    P(cp=1)=0.107  P(cp=2)=0.235  P(cp=3)=0.416  P(cp=4)=0.24

In [7]:
# ═══════════════════════════════════════════════════════════════
# PART 2 — SKLEARN IMPLEMENTATION  (GaussianNB on all features)
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PART 2 — NAIVE BAYES USING SKLEARN (GaussianNB)")
print("=" * 60)
 
sklearn_model = GaussianNB()
sklearn_model.fit(X_train, y_train)
y_pred_sklearn = sklearn_model.predict(X_test)
 
print(f"\nAccuracy  : {accuracy_score(y_test, y_pred_sklearn):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_sklearn,
                            target_names=["No Disease", "Disease"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_sklearn))
 
print("\n── sklearn Gaussian Parameters ──")
for i, feat in enumerate(FEATURES):
    mean_0, mean_1 = sklearn_model.theta_[0][i], sklearn_model.theta_[1][i]
    std_0  = np.sqrt(sklearn_model.var_[0][i])
    std_1  = np.sqrt(sklearn_model.var_[1][i])
    print(f"  {feat:10s}:  Class0 mean={mean_0:7.2f} std={std_0:6.2f}"
          f"  |  Class1 mean={mean_1:7.2f} std={std_1:6.2f}")
 
 
# ═══════════════════════════════════════════════════════════════
# PART 3 — COMPARISON TABLE
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("PART 3 — SIDE-BY-SIDE COMPARISON")
print("=" * 60)
 
from sklearn.metrics import precision_score, recall_score, f1_score
 
def metrics(y_true, y_pred, label):
    return {
        "Model"    : label,
        "Accuracy" : f"{accuracy_score(y_true, y_pred):.4f}",
        "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
        "Recall"   : f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
        "F1-Score" : f"{f1_score(y_true, y_pred, zero_division=0):.4f}",
    }
 
results = pd.DataFrame([
    metrics(y_test, y_pred_scratch, "From Scratch (Mixed NB)"),
    metrics(y_test, y_pred_sklearn, "sklearn GaussianNB"),
])
print(results.to_string(index=False))
 
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Key Differences:
  • From-Scratch model uses Gaussian PDF for numerical features
    (age, chol, thalach, oldpeak) and Laplace-smoothed categorical
    likelihoods for discrete features (sex, cp, exang, ca).
  • sklearn GaussianNB applies Gaussian PDF to ALL features
    (including categorical ones — less principled but fast).
  • The from-scratch model better respects the Bayesian Network
    feature types defined in the project report.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")


PART 2 — NAIVE BAYES USING SKLEARN (GaussianNB)

Accuracy  : 0.8195

Classification Report:
              precision    recall  f1-score   support

  No Disease       0.84      0.83      0.83       111
     Disease       0.80      0.81      0.80        94

    accuracy                           0.82       205
   macro avg       0.82      0.82      0.82       205
weighted avg       0.82      0.82      0.82       205

Confusion Matrix:
[[92 19]
 [18 76]]

── sklearn Gaussian Parameters ──
  age       :  Class0 mean=  52.49 std= 10.11  |  Class1 mean=  56.60 std=  7.53
  sex       :  Class0 mean=   0.56 std=  0.50  |  Class1 mean=   0.84 std=  0.37
  cp        :  Class0 mean=   2.79 std=  0.93  |  Class1 mean=   3.62 std=  0.80
  chol      :  Class0 mean= 242.58 std= 56.89  |  Class1 mean= 253.41 std= 48.02
  thalach   :  Class0 mean= 158.09 std= 19.30  |  Class1 mean= 138.97 std= 22.31
  exang     :  Class0 mean=   0.16 std=  0.36  |  Class1 mean=   0.56 std=  0.50
  oldpeak   :  Class0 